In [4]:
import pandas as pd
import joblib
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, roc_auc_score

train = pd.read_parquet('train.parquet')
val = pd.read_parquet('val.parquet')
scaler = joblib.load('scaler.pkl')
features = joblib.load('feature_list.pkl')

for df in [train, val]:
    df[features] = df[features].fillna(0)

X_train = scaler.transform(train[features])
X_val = scaler.transform(val[features])
y_train = train['is_late']
y_val = val['is_late']

FileNotFoundError: [Errno 2] No such file or directory: 'train.parquet'

In [ ]:
import mlflow

mlflow.set_experiment("late_delivery_prediction")
with mlflow.start_run():
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_metric("roc_auc", roc_auc_score(y_val, probs))
    mlflow.sklearn.log_model(model, "model")

ModuleNotFoundError: No module named 'mlflow'

In [ ]:
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
print("Baseline accuracy:", baseline.score(X_val, y_val))

Baseline accuracy: 0.9188909043793729


In [ ]:
model = LogisticRegression(class_weight='balanced', max_iter=1000)
model.fit(X_train, y_train)

preds = model.predict(X_val)
probs = model.predict_proba(X_val)[:, 1]

print(classification_report(y_val, preds))
print("ROC-AUC:", roc_auc_score(y_val, probs))

              precision    recall  f1-score   support

           0       0.93      0.67      0.78     14184
           1       0.10      0.42      0.16      1252

    accuracy                           0.65     15436
   macro avg       0.51      0.54      0.47     15436
weighted avg       0.86      0.65      0.73     15436

ROC-AUC: 0.56889380262871


In [ ]:
joblib.dump(model, 'model.pkl')
print("model saved!")

model saved!
